# pandas DataFrames: load, filter, group

You will load the taxi company's 12-trip log, keep only the night trips, and find which pickup district pays the best average fare, all day and at night.

About 10 minutes. Run the cells from top to bottom (Shift+Enter). No account or key is needed.

The outputs saved in this notebook come from pandas 3.1.1. If your pandas is older than 3.0, text columns show `object` instead of `str`, and a few object names look different. The numbers are the same.

## 1. Make the trip log file

The log is small, so it is written here as text and saved as `trips.csv` next to the notebook. It is a made-up log for this lesson, not real taxi data.

**Look for:** your pandas version, and the line saying the file was written.

In [1]:
import pandas as pd

CSV = """trip_id,pickup,dropoff,hour,minutes,fare
T01,Center,Harbor,8,14,11.5
T02,Airport,Center,9,15,12.0
T03,Harbor,Airport,11,28,20.0
T04,Center,Center,12,10,9.0
T05,Airport,Harbor,14,24,17.0
T06,Center,Center,17,9,8.5
T07,Harbor,Center,20,31,23.0
T08,Center,Airport,20,20,14.0
T09,Airport,Center,21,45,34.0
T10,Harbor,Center,22,36,25.5
T11,Center,Harbor,22,17,12.0
T12,Harbor,Airport,23,33,23.5
"""
with open("trips.csv", "w") as f:
    f.write(CSV)

print("pandas version:", pd.__version__)
print("wrote trips.csv,", CSV.count("\n") - 1, "trips")

pandas version: 3.1.1
wrote trips.csv, 12 trips


## 2. Load it into a DataFrame

`pd.read_csv` reads the file and returns a DataFrame: a table with named columns. The numbers on the left (0 to 11) are the row labels, the index.

**Look for:** `trips.shape` is `(12, 6)`: 12 rows, 6 columns.

In [2]:
trips = pd.read_csv("trips.csv")
print("trips.shape:", trips.shape)
print()
print(trips)

trips.shape: (12, 6)

   trip_id   pickup  dropoff  hour  minutes  fare
0      T01   Center   Harbor     8       14  11.5
1      T02  Airport   Center     9       15  12.0
2      T03   Harbor  Airport    11       28  20.0
3      T04   Center   Center    12       10   9.0
4      T05  Airport   Harbor    14       24  17.0
5      T06   Center   Center    17        9   8.5
6      T07   Harbor   Center    20       31  23.0
7      T08   Center  Airport    20       20  14.0
8      T09  Airport   Center    21       45  34.0
9      T10   Harbor   Center    22       36  25.5
10     T11   Center   Harbor    22       17  12.0
11     T12   Harbor  Airport    23       33  23.5


## 3. Each column has one type

In the file every value is text. `read_csv` looks at each column and gives the whole column one type.

**Look for:** `str` for trip_id, pickup and dropoff (`object` on pandas older than 3.0), `int64` for hour and minutes, `float64` for fare. The last line, `dtype: object`, describes the list itself, not a column. One column alone is a Series.

In [3]:
print("trips.dtypes:")
print(trips.dtypes)
print()
print('type(trips["fare"]):', type(trips["fare"]))

trips.dtypes:
trip_id        str
pickup         str
dropoff        str
hour         int64
minutes      int64
fare       float64
dtype: object

type(trips["fare"]): <class 'pandas.Series'>


## 4. A comparison makes a mask

One comparison runs over the whole `hour` column at once and gives one True or False per row.

**Look for:** 12 values; the first 6 are False and the last 6 are True. Night starts at 20:00, and `>=` includes 20 itself.

In [4]:
is_night = trips["hour"] >= 20
print('trips["hour"] >= 20:')
print(is_night)

trips["hour"] >= 20:
0     False
1     False
2     False
3     False
4     False
5     False
6      True
7      True
8      True
9      True
10     True
11     True
Name: hour, dtype: bool


## 5. The mask keeps rows

`trips[mask]` keeps the rows marked True. No loop and no `if`.

**Look for:** 6 night trips, T07 to T12. They keep their row labels 6 to 11.

In [5]:
night = trips[is_night]
print("night trips:", len(night))
print(night)

night trips: 6
   trip_id   pickup  dropoff  hour  minutes  fare
6      T07   Harbor   Center    20       31  23.0
7      T08   Center  Airport    20       20  14.0
8      T09  Airport   Center    21       45  34.0
9      T10   Harbor   Center    22       36  25.5
10     T11   Center   Harbor    22       17  12.0
11     T12   Harbor  Airport    23       33  23.5


## 6. Split-apply-combine: the average fare per district

`groupby("pickup")` splits the rows by pickup district, `["fare"]` picks the fare column in each group, and `.mean()` makes one number per group. Then we ask for the mean and the count side by side.

**Look for:** Airport 21.0, Center 11.0, Harbor 23.0. Harbor pays best over the whole day. The count column says 3, 5 and 4 trips.

In [6]:
print("average fare per pickup district, all day:")
print(trips.groupby("pickup")["fare"].mean())
print()
print("mean and count, all day:")
print(trips.groupby("pickup")["fare"].agg(["mean", "count"]))

average fare per pickup district, all day:
pickup
Airport    21.0
Center     11.0
Harbor     23.0
Name: fare, dtype: float64

mean and count, all day:
         mean  count
pickup              
Airport  21.0      3
Center   11.0      5
Harbor   23.0      4


## 7. Two conditions with `&`

Night trips that start in Harbor. Each condition makes its own mask, and `&` keeps a row only where both are True. Each condition sits in its own parentheses.

**Look for:** 3 rows: T07, T10 and T12.

In [7]:
harbor_night = trips[(trips["hour"] >= 20) & (trips["pickup"] == "Harbor")]
print("Harbor trips at night:", len(harbor_night))
print(harbor_night)

Harbor trips at night: 3
   trip_id  pickup  dropoff  hour  minutes  fare
6      T07  Harbor   Center    20       31  23.0
9      T10  Harbor   Center    22       36  25.5
11     T12  Harbor  Airport    23       33  23.5


## 8. Two broken lines, on purpose

These are the two wrong answers from the lesson. The cell catches each error and prints it, so the notebook keeps running.

**Look for:** the line with `and` raises a ValueError: "The truth value of a Series is ambiguous". The line with `&` but no parentheses raises a TypeError, because Python does `20 & trips["pickup"]` first.

In [8]:
try:
    trips[(trips["hour"] >= 20) and (trips["pickup"] == "Harbor")]
    print("with and: no error")
except Exception as e:
    print("with and:", type(e).__name__ + ":", e)

try:
    trips[trips["hour"] >= 20 & trips["pickup"] == "Harbor"]
    print("without parentheses: no error")
except Exception as e:
    print("without parentheses:", type(e).__name__ + ":", e)

with and: ValueError: The truth value of a Series is ambiguous. Use a.empty, a.item(), a.any() or a.all().
without parentheses: TypeError: unsupported operand type(s) for &: 'int' and 'StringArray'


## 9. The night-shift table

Filter first, then group. Always read the count next to the mean.

**Look for:** Airport on top with a mean of 34.0, but a count of 1. That one trip (T09) decides the whole number. Harbor's 24.0 rests on 3 trips.

In [9]:
print("night trips, mean and count per pickup district:")
print(night.groupby("pickup")["fare"].agg(["mean", "count"]))

night trips, mean and count per pickup district:
         mean  count
pickup              
Airport  34.0      1
Center   13.0      2
Harbor   24.0      3


## 10. The whole thing as one function

This is the function from the lesson's stepped code screen. It also prints what `groupby` returns before any function is applied.

**Look for:** `groups` and `fares` print as GroupBy objects with no numbers in them (the address after "at" is different on every run). The numbers appear only after `agg`. The returned table is sorted: Airport 34.0, Harbor 24.0, Center 13.0.

In [10]:
def night_fares(path):
    trips = pd.read_csv(path)
    is_night = trips["hour"] >= 20
    night = trips[is_night]
    groups = night.groupby("pickup")
    print("groups:", groups)
    fares = groups["fare"]
    print("fares: ", fares)
    table = fares.agg(["mean", "count"])
    table = table.sort_values(
        "mean", ascending=False)
    return table

table = night_fares("trips.csv")
print()
print("returned table:")
print(table)

groups: <pandas.api.typing.DataFrameGroupBy object at 0x10a295820>
fares:  <pandas.api.typing.SeriesGroupBy object at 0x10a296f30>

returned table:
         mean  count
pickup              
Airport  34.0      1
Harbor   24.0      3
Center   13.0      2


## 11. What the table costs in memory

`usecols` loads only the columns a question needs. `memory_usage(deep=True)` shows the bytes each column takes.

**Look for:** 3 columns with `usecols`. The pickup column takes several times more bytes than fare for the same 12 rows: text costs more than numbers. (Exact byte counts differ between pandas versions.)

In [11]:
small = pd.read_csv("trips.csv", usecols=["pickup", "hour", "fare"])
print("with usecols:")
print(small.dtypes)
print()
print("bytes per column:")
print(trips.memory_usage(deep=True))

with usecols:
pickup        str
hour        int64
fare      float64
dtype: object

bytes per column:
Index      132
trip_id    624
pickup     663
dropoff    663
hour        96
minutes     96
fare        96
dtype: int64


## Now change it

Each cell below has one line marked `# <- change this`. Edit that line, run the cell, and compare with the output saved here.

### Experiment 1: a later night shift

Change `20` to `22`. Only trips from 22:00 stay (T10, T11, T12). Airport has no trip left, so its row disappears from the table: a group with no rows has no mean. Harbor moves to the top with 24.5 from 2 trips.

In [12]:
NIGHT_FROM = 20  # <- change this
late = trips[trips["hour"] >= NIGHT_FROM]
print(f"night from {NIGHT_FROM}:00, {len(late)} trips")
print(late.groupby("pickup")["fare"].agg(["mean", "count"]))

night from 20:00, 6 trips
         mean  count
pickup              
Airport  34.0      1
Center   13.0      2
Harbor   24.0      3


### Experiment 2: the manager's second question

Change `"mean"` to `"max"` and `"fare"` to `"minutes"`. You get the longest trip per district: Airport 45, Center 20, Harbor 36. The split stays the same; only the column and the function applied to each group change.

In [13]:
COLUMN, FUNC = "fare", "mean"  # <- change this
print(FUNC, "of", COLUMN, "per pickup district:")
print(trips.groupby("pickup")[COLUMN].agg(FUNC))

mean of fare per pickup district:
pickup
Airport    21.0
Center     11.0
Harbor     23.0
Name: fare, dtype: float64


### Experiment 3: split by the other column

Change `"pickup"` to `"dropoff"`. Now each group holds the trips that ended in a district, and Airport comes first (about 19.17) instead of Harbor. The drivers asked where to wait for passengers, so pickup is the right split for their question.

In [14]:
SPLIT_BY = "pickup"  # <- change this
print("average fare per", SPLIT_BY, "district:")
print(trips.groupby(SPLIT_BY)["fare"].agg(["mean", "count"]))

average fare per pickup district:
         mean  count
pickup              
Airport  21.0      3
Center   11.0      5
Harbor   23.0      4


## What you should see

- `trips.shape` prints `(12, 6)`.
- `trips.dtypes` shows `str` for trip_id, pickup and dropoff (`object` if your pandas is older than 3.0), `int64` for hour and minutes, `float64` for fare.
- `trips["hour"] >= 20` prints 12 True/False values, and the last 6 are True.
- `trips.groupby("pickup")["fare"].mean()` gives Airport 21.0, Center 11.0, Harbor 23.0.
- The night table puts Airport first with a mean of 34.0 and a count of 1.
- The Harbor-at-night line with `&` but no parentheses raises a TypeError; the one with `and` raises a ValueError: "The truth value of a Series is ambiguous".